# 100 - Comparison of the DESC MAFs versus the E(B-V) cut of the WFD footprint

Notebooks 00-11 of this directory evaluate one DESC-related MAF each (notebook 00: the official effective-area summary of `ExgalM5WithCuts`) on the same 7 cadence simulations (`shrink_fp_dust_*` and `baseline`, all v5.3.6), whose WFD footprint is defined by an E(B-V) threshold of 0.050, 0.080, 0.120, 0.150, 0.199, 0.200 (baseline) and 0.250.

This notebook **does not run MAF**. It reads the per-run summary tables cached by those notebooks (`data_*` directories), reduces every MAF to one number per run, divides it by the value obtained for the baseline run (E(B-V) = 0.2), and draws all the resulting curves on the same axes. The legend says, for each MAF, whether it depends on `ExgalM5WithCuts` (legend group and line style):

| legend group | line style | meaning |
|---|---|---|
| **direct** | solid, filled markers | the MAF is `ExgalM5WithCuts`, wraps it, inherits from it, or is computed from its output |
| **indirect** | dash-dot, filled markers | uses a depth derived from `ExgalM5WithCuts` but restricted to another footprint |
| **re-implemented cuts** | dashed, open markers | applies the same kind of E(B-V) and depth cuts with its own code, without calling `ExgalM5WithCuts` |
| **none** | dotted, open markers | no cut inherited from `ExgalM5WithCuts` |

The dependency of each MAF is declared in Section 2 (`META`) with a one-line justification. Section 2.1 re-checks the declaration against the `rubin_sim` source when `rubin_sim` can be imported in the kernel.

**Outputs:** `data_100_MAFCOMPARISON/` (raw and normalized values), `figs_100_MAFCOMPARISON/` (PNG + PDF).

In [ ]:
import inspect
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

NB_TAG = "100_MAFCOMPARISON"
NB_DIR = Path.cwd()  # notebooks/10_DESCMAFDEPTHANDGALCOUNT
DATA_OUT = NB_DIR / f"data_{NB_TAG}"
FIGS_OUT = NB_DIR / f"figs_{NB_TAG}"
for d in (DATA_OUT, FIGS_OUT):
    d.mkdir(exist_ok=True)

BASELINE_EBV = 0.200
# E(B-V) threshold of each run -> run tag used in some cached file names
RUN_TAGS = {
    0.050: "shrink_fp_dust_0_050_v5_3_6_10yrs",
    0.080: "shrink_fp_dust_0_080_v5_3_6_10yrs",
    0.120: "shrink_fp_dust_0_120_v5_3_6_10yrs",
    0.150: "shrink_fp_dust_0_150_v5_3_6_10yrs",
    0.199: "shrink_fp_dust_0_199_v5_3_6_10yrs",
    0.200: "baseline_v5_3_6_10yrs",
    0.250: "shrink_fp_dust_0_250_v5_3_6_10yrs",
}

# Plot options
EXCLUDE = []  # curve keys to leave out (see the catalogue of Section 2)
YLIM = None  # e.g. (0.4, 1.2) to zoom on the main figure
print("Working directory:", NB_DIR)

In [ ]:
# Configuration générale des figures
plt.rcParams.update(
    {
        # Taille générale du texte
        "font.size": 16,
        # Titres
        "axes.titlesize": 20,
        "axes.titleweight": "bold",
        # Labels des axes
        "axes.labelsize": 18,
        # Chiffres graduations
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        # Légendes
        "legend.fontsize": 15,
        # Titre de la figure (suptitle)
        "figure.titlesize": 22,
    }
)

## 1 Readers of the cached summaries

Every reader returns a `pandas.Series` indexed by the E(B-V) threshold of the run (rounded to 3 decimals). Each one reads a table already written by the notebook that computed the MAF.

In [ ]:
def _ebv_index(values):
    return pd.Index(np.round(np.asarray(values, dtype=float), 3), name="E(B-V) threshold")


def from_summary(folder, csv_name, value_col, ebv_col="E(B-V) cut of the run"):
    """Per-run summary table (one row per run), e.g. ExgalM5WithCuts_per_run_summary.csv."""
    df = pd.read_csv(NB_DIR / folder / csv_name)
    return pd.Series(df[value_col].to_numpy(dtype=float), index=_ebv_index(df[ebv_col])).sort_index()


def from_year_table(folder, csv_name, value_col, year=10, ebv_col="ebv_threshold"):
    """Table with one row per (run, year), e.g. the 3x2pt FoM or the sigma8 bias: keep one year."""
    df = pd.read_csv(NB_DIR / folder / csv_name)
    df = df[df["year"] == year]
    return pd.Series(df[value_col].to_numpy(dtype=float), index=_ebv_index(df[ebv_col])).sort_index()


def from_tdc(csv_name, value_col):
    """TDC statistics table: run name in the first column, threshold in 'ebv_threshold'."""
    df = pd.read_csv(NB_DIR / "data_11_TDC" / csv_name, index_col=0)
    return pd.Series(df[value_col].to_numpy(dtype=float), index=_ebv_index(df["ebv_threshold"])).sort_index()


def from_kne(model, criterion, value_col="Total detected"):
    """One KNe_summary_<run>_*.csv per run: pick the (model, criterion) row."""
    out = {}
    for ebv, tag in RUN_TAGS.items():
        files = sorted((NB_DIR / "data_09_KNE").glob(f"KNe_summary_{tag}_*.csv"))
        if len(files) != 1:
            raise FileNotFoundError(f"expected one KNe_summary file for {tag}, found {len(files)}")
        df = pd.read_csv(files[0])
        row = df[(df["model"] == model) & (df["criterion"] == criterion)]
        out[ebv] = float(row[value_col].iloc[0])
    return pd.Series(out.values(), index=_ebv_index(list(out.keys()))).sort_index()


def normalize_to_baseline(series):
    """Divide by the value of the baseline run (threshold closest to BASELINE_EBV)."""
    gap = np.abs(series.index.to_numpy(dtype=float) - BASELINE_EBV)
    if gap.min() > 1e-6:
        raise ValueError("no baseline run in this series")
    return series / series.iloc[int(np.argmin(gap))]


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGS_OUT / f"{name}.{ext}", dpi=150, bbox_inches="tight")

## 2 Catalogue of the curves and dependency on `ExgalM5WithCuts`

`META` declares, for each MAF, its dependency category and the reason. `CURVES` lists the quantity plotted for each MAF, the notebook that produced it, and the reader that loads it. Extensive quantities are totals over the footprint of the run; intensive ones (depth, `zlim`) are means over the valid pixels. The FoM and the sigma8 bias are the year-10 values.

In [ ]:
DEP_ORDER = ["direct", "indirect", "reimpl", "none"]
DEP_STYLE = {
    "direct": dict(ls="-", lw=2.4, filled=True, short="direct", title="DEPENDS on ExgalM5WithCuts (solid)"),
    "indirect": dict(
        ls="-.",
        lw=1.9,
        filled=True,
        short="indirect",
        title="INDIRECT: depth derived from ExgalM5WithCuts (dash-dot)",
    ),
    "reimpl": dict(
        ls="--",
        lw=1.6,
        filled=False,
        short="re-implemented cuts",
        title="SAME CUTS re-implemented, no ExgalM5WithCuts (dashed)",
    ),
    "none": dict(
        ls=":", lw=1.8, filled=False, short="none", title="NO dependence on ExgalM5WithCuts (dotted)"
    ),
}

META = {
    "ExgalM5WithCuts": dict(
        dep="direct",
        why=(
            "Reference metric: coadded i-band depth, masked where E(B-V), 6-band coverage or depth cuts fail."
        ),
    ),
    "DepthLimitedNumGalMetric": dict(
        dep="direct",
        why=(
            "Nb 02 sets lim_ebv / nfilters_needed / lim_mag_i_ptsrc, which mirror extinction_cut / nFilters / "
            "depth_cut of ExgalM5WithCuts; confirm with the check of Section 2.1."
        ),
    ),
    "GalaxyCountsMetricExtended": dict(
        dep="none", why=("No E(B-V) cut in this study (README, nb 02): it follows the visits only.")
    ),
    "WeakLensingNvisits": dict(
        dep="reimpl",
        why=(
            "Own ebvlim and depth cut on top of the plain ExgalM5, no 6-band coverage requirement "
            "(legacy sims_maf code); confirm with Section 2.1."
        ),
    ),
    "RIZDetectionCoaddExposureTime": dict(
        dep="reimpl",
        why=(
            "Same module as WeakLensingNvisits with its own ebvlim; assumed to follow the same design, "
            "confirm with Section 2.1."
        ),
    ),
    "n_eff model (nb 04/05)": dict(
        dep="indirect",
        why=(
            "Nb 05: depth from ExgalM5WithCuts with all cuts disabled, restricted to the WL footprint of nb 03."
        ),
    ),
    "StaticProbesFoMEmulatorMetric": dict(
        dep="direct",
        why=("Summary metric fed by the ExgalM5WithCuts maps (depth and area), one map per year."),
    ),
    "TomographicClusteringSigma8biasMetric": dict(
        dep="direct",
        why=(
            "Nb 10: E(B-V) cut passed as extinction_cut of the parent metric; the footprint is also regenerated "
            "with dust_limit."
        ),
    ),
    "SNNSNMetric": dict(
        dep="none", why=("Own hard_dust_cut on the E(B-V) map; no coadded-depth cut from ExgalM5WithCuts.")
    ),
    "KNePopMetric": dict(
        dep="none", why=("No dust cut of its own (nb 09); same injected population for every run.")
    ),
    "TdcMetric": dict(
        dep="none", why=("No dust cut of its own; the E(B-V) footprint is a post-processing mask (nb 11).")
    ),
}

CURVES = [
    dict(
        key="exgal_area",
        metric="ExgalM5WithCuts",
        quantity="usable area [deg2]",
        nb="01",
        loader=lambda: from_summary(
            "data_01_EXGALM5CUTS", "ExgalM5WithCuts_per_run_summary.csv", "area_deg2"
        ),
    ),
    dict(
        key="exgal_depth",
        metric="ExgalM5WithCuts",
        quantity="mean i-band depth",
        nb="01",
        loader=lambda: from_summary("data_01_EXGALM5CUTS", "ExgalM5WithCuts_per_run_summary.csv", "mean"),
    ),
    dict(
        key="eff_area",
        metric="ExgalM5WithCuts",
        quantity="effective area, official batch [deg2]",
        nb="00",
        loader=lambda: from_summary(
            "data_00_EFFAREA",
            "EffectiveArea_official_per_run_summary.csv",
            "A_eff [deg2]",
            ebv_col="E(B-V) threshold",
        ),
    ),
    dict(
        key="dl_numgal",
        metric="DepthLimitedNumGalMetric",
        quantity="total galaxies",
        nb="02",
        loader=lambda: from_summary(
            "data_02_GALCOUNTS", "DepthLimitedNumGal_per_run_summary.csv", "total_galaxies"
        ),
    ),
    dict(
        key="gc_ext",
        metric="GalaxyCountsMetricExtended",
        quantity="total galaxies",
        nb="02",
        loader=lambda: from_summary(
            "data_02_GALCOUNTS", "GalaxyCountsMetricExtended_per_run_summary.csv", "total_galaxies"
        ),
    ),
    dict(
        key="wl_gri",
        metric="WeakLensingNvisits",
        quantity="total visits (gri)",
        nb="03",
        loader=lambda: from_summary("data_03_WL", "WL_NV_gri_per_run_summary.csv", "sum"),
    ),
    dict(
        key="wl_riz",
        metric="WeakLensingNvisits",
        quantity="total visits (riz)",
        nb="03",
        loader=lambda: from_summary("data_03_WL", "WL_NV_riz_per_run_summary.csv", "sum"),
    ),
    dict(
        key="wl_expt",
        metric="RIZDetectionCoaddExposureTime",
        quantity="total exposure time",
        nb="03",
        loader=lambda: from_summary("data_03_WL", "WL_EXPT_per_run_summary.csv", "sum"),
    ),
    dict(
        key="neff_generic",
        metric="n_eff model (nb 04/05)",
        quantity="N_eff, generic",
        nb="05",
        loader=lambda: from_summary(
            "data_05_NEFFMAPS", "neff_generic_per_run_summary.csv", "N_eff [1e6]", ebv_col="E(B-V) cut"
        ),
    ),
    dict(
        key="neff_r2",
        metric="n_eff model (nb 04/05)",
        quantity="N_eff, R2 cut",
        nb="05",
        loader=lambda: from_summary(
            "data_05_NEFFMAPS", "neff_R2cut_per_run_summary.csv", "N_eff [1e6]", ebv_col="E(B-V) cut"
        ),
    ),
    dict(
        key="fom3x2",
        metric="StaticProbesFoMEmulatorMetric",
        quantity="3x2pt FoM, year 10",
        nb="06",
        loader=lambda: from_year_table(
            "data_06_FOM3X2PTS", "FOM3x2pt_GP_all_runs_all_years.csv", "fom_3x2pt_GP"
        ),
    ),
    dict(
        key="nsn",
        metric="SNNSNMetric",
        quantity="total n_sn",
        nb="07",
        loader=lambda: from_summary("data_07_SNCOUNTS", "SNNSN_NSN_per_run_summary.csv", "sum"),
    ),
    dict(
        key="zlim",
        metric="SNNSNMetric",
        quantity="mean zlim",
        nb="07",
        loader=lambda: from_summary("data_07_SNCOUNTS", "SNNSN_ZLIM_per_run_summary.csv", "mean"),
    ),
    dict(
        key="kne",
        metric="KNePopMetric",
        quantity="detections (single model, multi_detect)",
        nb="09",
        loader=lambda: from_kne("single", "multi_detect"),
    ),
    dict(
        key="sigma8",
        metric="TomographicClusteringSigma8biasMetric",
        quantity="sigma8 bias, year 10 (lower = better)",
        nb="10",
        loader=lambda: from_year_table(
            "data_10_SIGMA8TOMO", "sigma8bias_all_runs_all_years.csv", "sigma8_bias"
        ),
    ),
    dict(
        key="tdc_nlens",
        metric="TdcMetric",
        quantity="number of lenses",
        nb="11",
        loader=lambda: from_tdc("TDC_stats_footprint_mask.csv", "n_lenses"),
    ),
    dict(
        key="tdc_dprec",
        metric="TdcMetric",
        quantity="distance precision (lower = better)",
        nb="11",
        loader=lambda: from_tdc("TDC_stats_footprint_mask.csv", "distance_precision_pct"),
    ),
]
CURVES = [c for c in CURVES if c["key"] not in EXCLUDE]

catalog = pd.DataFrame(
    [
        dict(
            key=c["key"],
            MAF=c["metric"],
            quantity=c["quantity"],
            notebook=c["nb"],
            dependency=DEP_STYLE[META[c["metric"]]["dep"]]["short"],
            why=META[c["metric"]]["why"],
        )
        for c in CURVES
    ]
)
pd.set_option("display.max_colwidth", 140)
display(catalog)

### 2.1 Optional check of the declared dependencies against the `rubin_sim` source

If `rubin_sim` can be imported in this kernel, each class is looked up and inspected: does it inherit from `ExgalM5WithCuts`, does its source mention `ExgalM5WithCuts`, does it call the plain `ExgalM5`? A mention found only in a docstring must be read by hand (`%psource <class>`). `NestedLinearMultibandModelMetric` is the parent metric of the sigma8 chain of notebook 10 and is checked as well.

In [ ]:
CHECK = [
    "ExgalM5WithCuts",
    "DepthLimitedNumGalMetric",
    "GalaxyCountsMetricExtended",
    "WeakLensingNvisits",
    "RIZDetectionCoaddExposureTime",
    "StaticProbesFoMEmulatorMetric",
    "NestedLinearMultibandModelMetric",
    "TomographicClusteringSigma8biasMetric",
    "SNNSNMetric",
    "KNePopMetric",
    "TdcMetric",
]


def find_class(name):
    import importlib
    import pkgutil
    import rubin_sim.maf as maf

    obj = getattr(maf, name, None)
    if inspect.isclass(obj):
        return obj
    for info in pkgutil.walk_packages(maf.__path__, prefix="rubin_sim.maf.", onerror=lambda n: None):
        try:
            mod = importlib.import_module(info.name)
        except Exception:
            continue
        obj = getattr(mod, name, None)
        if inspect.isclass(obj):
            return obj
    return None


try:
    import rubin_sim.maf  # noqa: F401
except ImportError:
    print("rubin_sim is not importable in this kernel: check skipped, the declared dependencies are used.")
else:
    rows = []
    for name in CHECK:
        cls = find_class(name)
        declared = META.get(name, {}).get("dep", "")
        if cls is None:
            rows.append(dict(class_name=name, found=False, declared=declared))
            continue
        try:
            src = inspect.getsource(cls)
        except (OSError, TypeError):
            src = ""
        rows.append(
            dict(
                class_name=name,
                found=True,
                declared=declared,
                module=cls.__module__,
                inherits_ExgalM5WithCuts="ExgalM5WithCuts" in [k.__name__ for k in cls.__mro__],
                source_mentions_ExgalM5WithCuts="ExgalM5WithCuts" in src,
                source_calls_plain_ExgalM5=bool(re.search(r"\bExgalM5\(", src)),
            )
        )
    display(pd.DataFrame(rows).set_index("class_name"))

## 3 Load the summaries and normalize to the baseline run

A curve whose cached file or column is missing is skipped and reported below; run the notebook that produces it first.

In [ ]:
raw, ratio, failed = {}, {}, {}
for c in CURVES:
    try:
        s = c["loader"]()
        raw[c["key"]] = s
        ratio[c["key"]] = normalize_to_baseline(s)
    except Exception as exc:
        failed[c["key"]] = f"{type(exc).__name__}: {exc}"

raw = pd.DataFrame(raw)
ratio = pd.DataFrame(ratio)
raw.to_csv(DATA_OUT / "raw_values_per_threshold.csv")
ratio.to_csv(DATA_OUT / "ratio_to_baseline_per_threshold.csv")

if failed:
    print("WARNING - curves skipped:")
    for k, msg in failed.items():
        print(f"  {k}: {msg}")
print(f"{ratio.shape[1]} curves loaded on {ratio.shape[0]} thresholds")
display(ratio.round(4))

## 4 All the MAFs on one plot

Ratio MAF / MAF(baseline) versus the E(B-V) threshold that defines the WFD footprint of the run. Colors identify the MAF, markers the quantity when a MAF has several; the legend is grouped by dependency on `ExgalM5WithCuts`.

In [ ]:
METRIC_ORDER = list(dict.fromkeys(c["metric"] for c in CURVES))
# 11 distinct colors, no gray (the order follows the catalogue of Section 2)
PALETTE = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#000000",
    "#bcbd22",
    "#17becf",
    "#ff00ff",
]
COLOR_OF = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(METRIC_ORDER)}
MARKERS = ["o", "s", "^", "D"]
for m in METRIC_ORDER:
    for j, c in enumerate([c for c in CURVES if c["metric"] == m]):
        c["marker"] = MARKERS[j % len(MARKERS)]


def curve_label(c):
    return f"{c['metric']} - {c['quantity']}"


def line_kwargs(c):
    st = DEP_STYLE[META[c["metric"]]["dep"]]
    col = COLOR_OF[c["metric"]]
    return dict(
        color=col,
        ls=st["ls"],
        lw=st["lw"],
        marker=c["marker"],
        ms=6.5,
        mfc=col if st["filled"] else "white",
        mew=1.4,
    )


def draw_curves(ax, curves):
    for c in curves:
        if c["key"] in ratio:
            y = ratio[c["key"]]
            ax.plot(y.index, y.values, **line_kwargs(c))
    ax.axhline(1.0, color="k", lw=0.8, alpha=0.6)
    ax.axvline(BASELINE_EBV, color="k", lw=0.8, alpha=0.6, ls=(0, (1, 3)))
    ax.set_xticks([0.05, 0.08, 0.12, 0.15, 0.20, 0.25])
    ax.set_xlabel("E(B-V) threshold defining the WFD footprint of the run")
    ax.grid(alpha=0.3)


def grouped_legend(ax, curves, **kwargs):
    handles, labels, headers = [], [], []
    for dep in DEP_ORDER:
        members = [c for c in curves if c["key"] in ratio and META[c["metric"]]["dep"] == dep]
        if not members:
            continue
        handles.append(Line2D([], [], linestyle="none", marker="none"))
        labels.append(DEP_STYLE[dep]["title"])
        headers.append(DEP_STYLE[dep]["title"])
        for c in members:
            handles.append(Line2D([], [], **line_kwargs(c)))
            labels.append(curve_label(c))
    leg = ax.legend(handles, labels, frameon=False, handlelength=3.2, labelspacing=0.45, **kwargs)
    for t in leg.get_texts():
        if t.get_text() in headers:
            t.set_fontweight("bold")
    return leg


fig, ax = plt.subplots(figsize=(14, 8))
draw_curves(ax, CURVES)
ax.set_ylabel("MAF value / MAF value of the baseline run (E(B-V) = 0.2)")
ax.set_title("DESC MAFs versus the E(B-V) cut of the WFD footprint, normalized to the baseline")
if YLIM is not None:
    ax.set_ylim(*YLIM)
grouped_legend(ax, CURVES, loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0, fontsize=14)
save_fig(fig, "MAFDESC_ratio_to_baseline_vs_ebv")
plt.show()

### 4.1 Same curves, one panel per dependency category

Easier to read when the curves overlap on the main figure; every panel has its own y-range.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, dep in zip(axes.ravel(), DEP_ORDER):
    members = [c for c in CURVES if META[c["metric"]]["dep"] == dep]
    draw_curves(ax, members)
    ax.set_title(DEP_STYLE[dep]["title"], fontsize=10)
    if any(c["key"] in ratio for c in members):
        ax.legend(
            [Line2D([], [], **line_kwargs(c)) for c in members if c["key"] in ratio],
            [curve_label(c) for c in members if c["key"] in ratio],
            fontsize=7.5,
            frameon=False,
            handlelength=3.0,
        )
for ax in axes[:, 0]:
    ax.set_ylabel("MAF / MAF(baseline)")
for ax in axes[0, :]:
    ax.set_xlabel("")
fig.suptitle("DESC MAFs versus the E(B-V) cut, grouped by dependency on ExgalM5WithCuts", y=0.995)
fig.tight_layout()
save_fig(fig, "MAFDESC_ratio_to_baseline_vs_ebv_by_dependency")
plt.show()

### 4.2 Total effective number of galaxies N_eff,tot versus the E(B-V) cut

Dedicated view of the `n_eff model (nb 04/05)` curves of Section 4. For each run, the total effective number of galaxies is the integral of the n_eff map (galaxies / arcmin2) over the angular footprint of the run:

$$N_{\rm eff,tot} = \int_{\rm footprint} n_{\rm eff}\, d\Omega = \langle n_{\rm eff}\rangle \times A_{\rm footprint}$$

with $A$ in arcmin2 (1 deg2 = 3600 arcmin2). It is recomputed here from the mean n_eff and the area cached by notebook 05 and cross-checked against the `N_eff [1e6]` column. Its ratio to the baseline (E(B-V) = 0.2) factorizes exactly as

$$\frac{N_{\rm eff,tot}}{N_{\rm eff,tot}^{\rm baseline}} = \frac{A}{A^{\rm baseline}} \times \frac{\langle n_{\rm eff}\rangle}{\langle n_{\rm eff}\rangle^{\rm baseline}},$$

so the figure shows N_eff,tot, its normalization to the baseline together with the area ratio, and the mean-n_eff ratio, to separate the footprint effect from the per-pixel effect. Both n_eff variants of notebook 05 are drawn (generic, and with the R2 cut). Depth comes from `ExgalM5WithCuts` (all cuts disabled) restricted to the WL footprint of notebook 03, hence the *indirect* dependency declared in Section 2.

In [ ]:
ARCMIN2_PER_DEG2 = 3600.0
NEFF_VARIANTS = {
    "generic": dict(
        csv="neff_generic_per_run_summary.csv", label="n_eff generic", color="#2ca02c", marker="o"
    ),
    "R2cut": dict(
        csv="neff_R2cut_per_run_summary.csv", label="n_eff with R2 cut", color="#8c564b", marker="s"
    ),
}


def neff_tot_table(csv_name):
    """N_eff,tot = integral of n_eff over the angular footprint, one row per run (E(B-V) threshold)."""
    kw = dict(ebv_col="E(B-V) cut")
    area = from_summary("data_05_NEFFMAPS", csv_name, "area_deg2", **kw)
    mean_neff = from_summary("data_05_NEFFMAPS", csv_name, "mean n_eff", **kw)
    cached = from_summary("data_05_NEFFMAPS", csv_name, "N_eff [1e6]", **kw)
    integral = mean_neff * area * ARCMIN2_PER_DEG2 / 1e6  # [1e6 galaxies]
    dev = float(np.max(np.abs(integral / cached - 1.0)))
    if dev > 1e-3:
        print(f"WARNING {csv_name}: <n_eff> x area differs from the cached N_eff by {dev:.2e}")
    return pd.DataFrame(
        {
            "area_deg2": area,
            "mean_neff_per_arcmin2": mean_neff,
            "Neff_tot_1e6": integral,
            "area_ratio": normalize_to_baseline(area),
            "mean_neff_ratio": normalize_to_baseline(mean_neff),
            "Neff_tot_ratio": normalize_to_baseline(integral),
        }
    )


neff_tot = {}
for name, v in NEFF_VARIANTS.items():
    try:
        neff_tot[name] = neff_tot_table(v["csv"])
    except Exception as exc:
        print(f"N_eff,tot ({name}) skipped: {type(exc).__name__}: {exc}")

if neff_tot:
    pd.concat(neff_tot, names=["n_eff variant"]).to_csv(DATA_OUT / "Neff_tot_vs_ebv.csv")
    display(pd.concat(neff_tot, names=["n_eff variant"]).round(4))

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    ax_raw, ax_norm, ax_mean = axes
    for name, tab in neff_tot.items():
        v = NEFF_VARIANTS[name]
        kw = dict(color=v["color"], marker=v["marker"], ms=7, lw=2.0)
        ax_raw.plot(tab.index, tab["Neff_tot_1e6"], label=v["label"], **kw)
        ax_norm.plot(tab.index, tab["Neff_tot_ratio"], label=f"N_eff,tot - {v['label']}", **kw)
        ax_mean.plot(tab.index, tab["mean_neff_ratio"], label=v["label"], **kw)
    # the area is common to both variants: draw it once
    first = next(iter(neff_tot.values()))
    ax_norm.plot(
        first.index,
        first["area_ratio"],
        color="k",
        ls="--",
        lw=1.8,
        marker="^",
        mfc="white",
        ms=6,
        label="footprint area (same for both)",
    )
    ax_raw.set_ylabel(r"$N_{\rm eff,tot}=\int n_{\rm eff}\,d\Omega$  [$10^6$ galaxies]")
    ax_raw.set_title("Total effective number of galaxies", fontsize=16)
    ax_norm.set_ylabel("value / value of the baseline run (E(B-V) = 0.2)")
    ax_norm.set_title("N_eff,tot normalized to the baseline", fontsize=16)
    ax_mean.set_ylabel(r"$\langle n_{\rm eff}\rangle$ / baseline")
    ax_mean.set_title("Mean n_eff per pixel (footprint effect removed)", fontsize=16)
    for ax in axes:
        ax.set_xticks([0.05, 0.08, 0.12, 0.15, 0.20, 0.25])
        ax.set_xlabel("E(B-V) threshold defining the WFD footprint of the run")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, fontsize=12)
    for ax in (ax_norm, ax_mean):
        ax.axhline(1.0, color="k", lw=0.8, alpha=0.6)
        ax.axvline(BASELINE_EBV, color="k", lw=0.8, alpha=0.6, ls=(0, (1, 3)))
    fig.suptitle("Total effective galaxy number integrated over the footprint versus the E(B-V) cut", y=1.0)
    fig.tight_layout()
    save_fig(fig, "Neff_tot_vs_ebv_and_ratio_to_baseline")
    plt.show()

### 4.3 Effective area of the official batch (notebook 00)

Notebook 00 reuses the *Cosmology / Static Science* block of `science_radar_batch` without writing a new metric: the effective area is the summary statistic `CountRatioMetric(norm_val=1/pix_area)` attached to `ExgalM5WithCuts`, i.e. A_eff = (number of valid pixels) x (pixel area) in deg2 (the batch label says `deg` but the unit is deg2). It is therefore a **direct** dependency of `ExgalM5WithCuts` and must coincide with the usable area of notebook 01 when both use the same cuts (i band, 6 filters, depth cut 25.9, `extinction_cut` = threshold of the run, nside 64); the cell below checks it.

The right panel compares its ratio to the baseline with the area of the gri weak-lensing footprint (notebook 05) and with N_eff,tot of Section 4.2, to show how much of the N_eff,tot trend is already contained in a purely geometric area.

In [ ]:
# Effective area of the official batch (notebook 00) = ExgalM5WithCuts + CountRatioMetric summary
if "eff_area" not in ratio:
    print("Effective area (notebook 00) not loaded: run notebook 00 first.")
else:
    a_eff = raw["eff_area"]
    tab = pd.DataFrame({"A_eff_deg2": a_eff, "A_eff_ratio": ratio["eff_area"]})
    if "exgal_area" in raw:  # cross-check against the usable area of notebook 01
        tab["area_nb01_deg2"] = raw["exgal_area"]
        tab["A_eff_over_area_nb01"] = a_eff / raw["exgal_area"]
        dev = float((tab["A_eff_over_area_nb01"] - 1.0).abs().max())
        print(f"max |A_eff (nb 00) / area (nb 01) - 1| = {dev:.2e}")
        if dev > 1e-3:
            print("WARNING: notebooks 00 and 01 differ (n_filters, depth_cut, extinction_cut or nside)")
    if neff_tot:  # weak-lensing footprint and N_eff,tot of Section 4.2
        first = next(iter(neff_tot.values()))
        tab["WL_area_ratio_nb05"] = first["area_ratio"]
        for name, t in neff_tot.items():
            tab["Neff_tot_ratio_" + name] = t["Neff_tot_ratio"]
    tab.to_csv(DATA_OUT / "EffectiveArea_official_vs_ebv.csv")
    display(tab.round(4))

    fig, (ax_raw, ax_norm) = plt.subplots(1, 2, figsize=(15, 6))
    ax_raw.plot(
        a_eff.index, a_eff.values, "o-", color="tab:blue", lw=2.2, ms=7, label="A_eff, official batch (nb 00)"
    )
    if "exgal_area" in raw:
        ax_raw.plot(
            raw["exgal_area"].index,
            raw["exgal_area"].values,
            "x",
            color="k",
            ms=11,
            mew=2,
            label="usable area, ExgalM5WithCuts (nb 01)",
        )
    ax_raw.set_ylabel("Effective area [deg$^2$]")
    ax_raw.set_title("A_eff, ExgalM5WithCuts + CountRatioMetric", fontsize=15)

    r = ratio["eff_area"]
    ax_norm.plot(r.index, r.values, "o-", color="tab:blue", lw=2.2, ms=7, label="A_eff (nb 00, 6 bands)")
    for xi, yi in zip(r.index, r.values):
        ax_norm.annotate(
            f"{yi:.3f}", (xi, yi), textcoords="offset points", xytext=(0, -16), ha="center", fontsize=10
        )
    if "WL_area_ratio_nb05" in tab:
        ax_norm.plot(
            tab.index,
            tab["WL_area_ratio_nb05"],
            "^--",
            color="k",
            mfc="white",
            lw=1.6,
            label="WL footprint area (nb 05, gri)",
        )
    for name in neff_tot:
        v = NEFF_VARIANTS[name]
        ax_norm.plot(
            tab.index,
            tab["Neff_tot_ratio_" + name],
            ls=":",
            color=v["color"],
            marker=v["marker"],
            mfc="white",
            lw=1.8,
            label="N_eff,tot, " + v["label"],
        )
    ax_norm.axhline(1.0, color="k", lw=0.8, alpha=0.6)
    ax_norm.axvline(BASELINE_EBV, color="k", lw=0.8, alpha=0.6, ls=(0, (1, 3)))
    ax_norm.set_ylabel("value / value of the baseline run (E(B-V) = 0.2)")
    ax_norm.set_title("Normalized to the baseline (10 yr)", fontsize=15)
    for ax in (ax_raw, ax_norm):
        ax.set_xticks([0.05, 0.08, 0.12, 0.15, 0.20, 0.25])
        ax.set_xlabel("E(B-V) threshold defining the WFD footprint of the run")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, fontsize=12)
    fig.suptitle("Effective area of the official batch versus the E(B-V) cut", y=1.0)
    fig.tight_layout()
    save_fig(fig, "EffArea_official_vs_ebv_and_ratio_to_baseline")
    plt.show()

## 5 Summary table

Ratio at the smallest and the largest threshold, and the largest departure from the baseline over all the runs.

In [ ]:
lo, hi = ratio.index.min(), ratio.index.max()
rows = []
for c in CURVES:
    if c["key"] not in ratio:
        continue
    r = ratio[c["key"]]
    rows.append(
        {
            "dependency on ExgalM5WithCuts": DEP_STYLE[META[c["metric"]]["dep"]]["short"],
            "MAF": c["metric"],
            "quantity": c["quantity"],
            "notebook": c["nb"],
            f"ratio @ {lo:.3f}": r.loc[lo],
            f"ratio @ {hi:.3f}": r.loc[hi],
            "max |ratio - 1|": (r - 1.0).abs().max(),
        }
    )
summary = pd.DataFrame(rows)
order = {DEP_STYLE[d]["short"]: i for i, d in enumerate(DEP_ORDER)}
summary = summary.sort_values(
    "dependency on ExgalM5WithCuts", key=lambda s: s.map(order), kind="stable"
).reset_index(drop=True)
summary.to_csv(DATA_OUT / "summary_ratio_to_baseline.csv", index=False)
display(summary.round(3))

## 6 Reading guide and caveats

- The ratio is a ratio of *totals* for extensive quantities (galaxy counts, visits, exposure time, `n_sn`, `N_eff`, number of lenses, detections, footprint area, effective area A_eff) and of *means* for intensive ones (`ExgalM5WithCuts` depth, `zlim`). The FoM and the sigma8 bias are the year-10 values. The ratio mixes the change of the footprint (pixels gained or lost when the threshold moves) and the change of the per-pixel value produced by the scheduler.
- The 0.199 and 0.200 runs have almost the same threshold but different visit distributions (the baseline file has 11 years and is truncated to the first 10); their difference gives an idea of the scheduler-to-scheduler scatter.
- The sigma8 bias and the TDC distance precision are better when lower; all the other quantities are better when higher.
- `KNePopMetric` counts detections of one injected population (fixed seed); with about 5000 detections the Poisson noise is about 1.4 %, so its small wiggles are noise.
- The resolution differs between notebooks (`nside` 64, 128 and 32). Ratios of totals are little affected; per-pixel means would be.
- The dependency categories are declarations (Section 2) checked against the source only if `rubin_sim` is importable (Section 2.1).
- The effective area A_eff of notebook 00 is the official-batch statistic: number of pixels that pass the E(B-V), 6-band and depth cuts of `ExgalM5WithCuts` times the pixel area (nside 64). It is purely geometric and requires ugrizy coverage, so it is smaller than the gri weak-lensing footprint of notebook 03 used for N_eff,tot in Section 4.2. Its curve must coincide with the usable-area curve of notebook 01 (same cuts); Section 4.3 checks it. The seeing-weighted effective area of notebook 00b is not read here.